# DAEN 328, Fall 2026
## Assignment 1: NYC COVID-19 Outcomes, from API to SQLite
**Due Friday, September 11, 11:59 pm. 100 points. The grading rubric is posted on Canvas.**

In this assignment you will build the complete workflow of:
1. Fetching data from a public API.
2. Cleaning and transforming raw JSON data into a structured format.
3. Performing quality checks to validate data consistency.
4. Storing the cleaned data in an SQLite database for future analysis.

## Objective:
Analyze COVID-19 testing and outcomes data to gain insights and save it in a reusable format for further exploration and analysis.


## Step 1: Import Required Libraries

To start, we need to import the following libraries:
- `requests`: To fetch data from the API.
- `pandas`: To manipulate and clean tabular data.
- `sqlite3`: To store data in a lightweight SQLite database.
- `json`: To process JSON data, the format of the API response.
- `matplotlib.pyplot`: To visualize data during quality checks.
- `os`: Setting the working directory to specifyied paths, other operating system functions


These libraries provide all the tools needed for the project.


In [1]:
# Always include these two lines.
# They allow multiple cell outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%pip install requests pandas matplotlib

# Importing required libraries
import requests  # To make HTTP requests and fetch data from APIs
import pandas as pd  # To store, manipulate, and clean tabular data
import sqlite3  # To interact with an SQLite database for data storage
import json  # To handle JSON data from APIs
import matplotlib.pyplot as plt  # Optional, for data visualization
import os # working with operating system functions


# Display confirmation
print("Libraries imported successfully!")

  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 939.7 kB/s eta 0:00:10
   -- ------------------------------------- 0.5/9.5 MB 939.7 kB/s eta 0:00:10
   --- ------------------------------------ 0.8/9.5 MB 912.5 kB/s eta 0:00:10
   ---- ----------------------------------- 1.0/9.5 MB 894.2 kB/s eta 0:00:10
   ---- ----------------------------------- 1.0/9.5 MB 894.2 kB/s eta 0:00:10
   ---- ----------------------------------- 1.0/9.5 MB 894.2 kB/s eta 0:00:10
   ----- ---------------------------------- 1.3/9.5 M

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully!


#### Look at your working directory and set it to the directory you want

In [2]:
import os

# All files for this assignment live in a "data" folder next to this notebook.
# This works the same on Windows, macOS, and Linux. No personal paths needed.
directory = os.path.join(os.getcwd(), "data")
os.makedirs(directory, exist_ok=True)
print(f"Working data folder: {directory}")


Working data folder: c:\Users\danie\OneDrive\Desktop\College Stuff\DAEN 328\Assigment 1\data


## Step 2: Fetch Data from the NYC COVID-19 Outcomes API
In this step, we define a function to fetch data from an API and make a call to retrieve data.

### Data Portal URL
The dataset used here is the [NYC COVID-19 Outcomes by Testing Cohorts](https://data.cityofnewyork.us/Health/COVID-19-Outcomes-by-Testing-Cohorts-Cases-Hospita/cwmx-mvra/about_data).

1. Download the User-Guide and Data dictionary from the attachments and explore. These files describe the data and how to use it. Read them carefully and pay attention to their structure. They provide a decent template for how to structure a data dictionary and data users guide.

2. Explore the Actions and Export Buttons on the top-right corner.
3. Look at the Data tab at the upper-left. 

Also, read about the JSON data format here: 
1. https://www.digitalocean.com/community/tutorials/an-introduction-to-json
2. https://www.freecodecamp.org/news/how-to-use-the-json-module-in-python/



### API URL
To get the data, we will use the API endpoint:  
`https://data.cityofnewyork.us/resource/cwmx-mvra.json`

- How did we find this API?
- What does API stand for?

### Steps
1. We create a reusable function `fetch_api_data()` to make API calls.
2. Implement pagination using `$limit` and `$offset` to fetch all records. 
3. Fetch the data in batches to overcome API limitations. (Why is this necessary?)
4. Validates the API response and handle errors. Also, save data to a json file as it is retrieved.
5. Combine all batches into a single dataset for further processing.
6. After fetching, display the first few records to inspect the structure of the data.

In [ ]:
def fetch_api_data(api_url, output_file, batch_size=1000, num_records=None):
    """
    Fetches all data from the API in chunks using $limit and $offset parameters, 
    and saves each batch to a file incrementally.

    Parameters:
    - api_url (str): The base URL of the API.
    - output_file (str): Path to the JSON file to save data incrementally.
    - batch_size (int): Number of records to fetch per request (default: 1000).
    - num_records (int or None): Maximum number of records to fetch. If None, fetch all records.
    """
    offset = 0
   
    # Check if the output file already exists and load existing data
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            try:
                all_data = json.load(f)
                print(f"Resuming from {len(all_data)} records in {output_file}.")
            except json.JSONDecodeError:
                print(f"{output_file} is corrupted or empty. Starting fresh.")
                all_data = []
    else:
        all_data = []

    # Calculate the starting offset based on the existing data
    offset = len(all_data)
    print(f"Starting from offset {offset}...")

    while True:
        # Add $limit and $offset parameters to the API URL
        paginated_url = f"{api_url}?$limit={batch_size}&$offset={offset}"
        print(f"Fetching records starting at offset {offset}...")
        
        # Fetch data from the API
        try:
            response = requests.get(paginated_url)
            response.raise_for_status()
            batch_data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break

        # Stop if no more data is returned
        if not batch_data:
            print("No more data to fetch.")
            break

        # Append the batch to the combined data list
        all_data.extend(batch_data)

        # Save the updated data to the output file incrementally
        with open(output_file, "w") as f:
            json.dump(all_data, f, indent=2)
        print(f"Appended {len(batch_data)} records. Total records saved: {len(all_data)}")

        # Update offset to fetch the next batch
        offset += batch_size

        # Stop if a specific number of records is requested and reached
        if num_records is not None and len(all_data) >= num_records:
            print(f"Reached the specified number of records: {num_records}.")
            break

        # Break if the batch size is less than the limit, indicating the end of the dataset
        if len(batch_data) < batch_size:
            print("Reached the end of the dataset.")
            break

    print(f"Fetched a total of {len(all_data)} records. Data saved to {output_file}.")
    return all_data




> **Important, before re-running the fetch:** the function above *resumes* from `api_data.json` if it already exists. If a previous run was interrupted or you changed any settings, **delete `data/api_data.json` first**, otherwise you'll get a mix of old and new data and your row counts will be wrong.


### Explanation of the Code
1. **Function Definition**:  
   - `fetch_api_data(api_url)`: Takes the API endpoint URL as input.
   - Makes an HTTP GET request using `requests.get()`.
   - Checks for successful responses using `response.raise_for_status()` to handle errors like 404 or 500.
   - Parses the JSON response using `response.json()` and returns it.

2. **API Call**:  (see cell below)
   - We call the `fetch_api_data()` function with the NYC COVID-19 API URL.
   - The fetched data is stored in the `api_data_all` variable.

3. **Output**:  
   - The total number of records fetched.
   - A sample of the first 5 records is displayed in a readable format using `json.dumps()`.


Run the code below to execute the function and fetch the data. Note that this might take some time, so be patient. 




In [ ]:
# API URL for NYC COVID-19 Outcomes dataset
api_url = "https://data.cityofnewyork.us/resource/cwmx-mvra.json"

# The JSON file will be saved inside the data folder created earlier
json_file_path = os.path.join(directory, "api_data.json")

# Fetch the data.
# num_records=None means fetch ALL records (~176,000+, takes about 2 minutes).
# Do NOT set a fixed cap like 100000. The dataset keeps growing, and a cap
# silently cuts off the most recent dates (which Question 9 depends on).
api_data = fetch_api_data(
    api_url=api_url,
    output_file=json_file_path,
    batch_size=1000,
    num_records=None,
)

# Verify the total number of records fetched
print(f"Total records fetched: {len(api_data)}")

# Display a sample of the data to inspect
if api_data:
    print("Sample data (first 5 records):")
    print(json.dumps(api_data[:5], indent=2))


## Step 3: Load JSON Data to Pandas DataFrame and Inspect Dataset

After fetching the raw JSON data:
1. Convert it into a structured format using `pandas.DataFrame()`.
2. Inspect the structure of the DataFrame using:
   - `.head()`: To view the first few rows.
   - `.info()`: To check column data types and counts.
   - `.describe()`: To get summary statistics for numeric columns.

This step prepares the data for cleaning and analysis.


In [ ]:

# Read the JSON file into a DataFrame
df = pd.read_json(json_file_path)

# Display the first few rows of the DataFrame
print(df.head())

# Display the first few rows to inspect the structure
print("Sample DataFrame (first 5 rows):")
print(df.head())

# Display information about the DataFrame's structure and data types
print("\nDataFrame Info:")
print(df.info())

# Display summary statistics for numeric columns
print("\nSummary Statistics for Numeric Columns:")
print(df.describe())


# Additional explanation:
# 1. pd.read_csv(csv_file_path): opens csv file into a pandas Dataframe.
# 2. df.head(): Displays the first 5 rows of the DataFrame for a quick overview.
# 3. df.info(): Shows column names, data types, and non-null counts.
# 4. df.describe(): Provides basic statistics (e.g., mean, min, max) for numeric columns.


### Observations:
1. **Data Columns**:
   - `extract_date`: The date when data was extracted (string format).
   - `specimen_date`: The date of specimen collection (string format).
   - `number_tested`, `number_confirmed`, `number_hospitalized`, `number_deaths`: Numeric data representing COVID-19 testing and outcomes.

2. **Data Issues**:
   - Date columns are in string format and need to be converted to datetime.
   - Numeric columns may have been imported as strings and need conversion.
   - Missing or null values should be addressed during the cleaning step.

3. **Next Step**:
   - Clean the data to fix these issues and ensure it’s ready for analysis and storage.


## Step 4: Data Cleaning

To prepare the data for analysis, we will clean the dataset as follows:

### 1. Convert Date Columns to `datetime`
- `extract_date` and `specimen_date` are currently in string (`object`) format.
- We'll convert these to `datetime` using `pd.to_datetime()` for easier manipulation.
- Invalid or out-of-range dates will be handled using `errors='coerce'`, which replaces them with `NaT` (Not a Time).

### 2. Convert Numeric Columns to Proper Types
- Columns like `number_tested`, `number_confirmed`, etc., are currently in string (`object`) format.
- These will be converted to numeric types using `pd.to_numeric()`.
- Any invalid or non-numeric entries will be coerced into `NaN`.

### 3. Handle Missing Values
- After conversions, missing values (`NaN` for numbers and `NaT` for dates) will be replaced with `0`.
- This ensures the data is ready for further analysis.

### 4. Verify the Cleaned Data
- We'll inspect the cleaned dataset using:
  - `.info()`: To check column types and non-null counts.
  - `.head()`: To view the first few rows.


In [ ]:
# Step 1: Convert date columns to datetime
df['extract_date'] = pd.to_datetime(df['extract_date'], format="%Y-%m-%dT%H:%M:%S.%f", errors='coerce')
df['specimen_date'] = pd.to_datetime(df['specimen_date'], errors='coerce')

# Step 2: Handle missing datetime values
# Replace missing dates with a placeholder (e.g., '1970-01-01')
df['extract_date'] = df['extract_date'].fillna(pd.Timestamp("1970-01-01"))
df['specimen_date'] = df['specimen_date'].fillna(pd.Timestamp("1970-01-01"))

# Step 3: Convert numeric columns to appropriate types
numeric_columns = ['number_tested', 'number_confirmed', 'number_hospitalized', 'number_deaths']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Replace missing numeric values with 0
df[numeric_columns] = df[numeric_columns].fillna(0)

# Step 4: Prepare for SQLite storage
# Convert 'datetime64[ns]' columns to strings in ISO 8601 format for compatibility with SQLite
df['extract_date'] = df['extract_date'].dt.strftime("%Y-%m-%d")
df['specimen_date'] = df['specimen_date'].dt.strftime("%Y-%m-%d")

# Step 5: Verify the cleaned and prepared data
print("Cleaned and Prepared DataFrame Info:")
print(df.info())

# Display the first few rows of the cleaned DataFrame to verify the structure and data
print("\nSample of Cleaned and Prepared DataFrame (first 5 rows):")
df.head()




## Observations After Cleaning

1. **Invalid `specimen_date` Entries**:
   - All `specimen_date` values were successfully converted to `datetime`, with no invalid entries.

2. **Cleaned DataFrame Info**:
   - **Row Count**: Matches the number of rows in the dataset after cleaning.
   - **Column Types**:
     - `extract_date` and `specimen_date`: Converted to `datetime64[ns]`.
     - `number_tested`, `number_confirmed`, `number_hospitalized`, `number_deaths`: Converted to numeric types (`int64`).
   - No missing values remain in the dataset.

3. **Sample Data**:
   - The dataset is clean, with dates and numeric values properly formatted.

### Next Steps:
1. **Perform Quality Checks**:
   - Check for duplicate rows in the dataset.
   - Verify that there are no negative values in numeric columns.
   - Visualize numeric columns using a boxplot to identify any potential outliers.

2. **Store the Cleaned Data**:
   - Once quality checks are complete, store the cleaned dataset in an SQLite database for further analysis or use.


## Step 5: Perform Quality Checks

Quality checks ensure that the cleaned data is consistent, valid, and ready for analysis. In this step, we will:

1. **Check for Duplicate Rows**:
   - Count and remove duplicate rows if any exist.

2. **Check for Negative Values**:
   - Verify that numeric columns (`number_tested`, `number_confirmed`, `number_hospitalized`, `number_deaths`) do not contain negative values.

3. **Visualize Numeric Columns**:
   - Use boxplots to identify potential outliers or anomalies in the numeric columns.

These checks help validate the integrity of the dataset.


In [ ]:
# Step 1: Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# If duplicates exist, drop them
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates removed.")

# Step 2: Check for negative values in numeric columns
negative_values = (df[['number_tested', 'number_confirmed', 'number_hospitalized', 'number_deaths']] < 0).sum()
print("\nNegative value counts in numeric columns:")
print(negative_values)

# Step 3: Visualize numeric columns for outliers
print("\nVisualizing numeric data distributions...")
df[['number_tested', 'number_confirmed', 'number_hospitalized', 'number_deaths']].boxplot(figsize=(10, 6))
plt.title("Boxplot of Numeric Columns")
plt.ylabel("Values")
plt.show()

# Step 4: Quality Summary
quality_summary = {
    "Total Rows": len(df),
    "Total Duplicates": duplicates,
    "Negative Values": negative_values.to_dict()
}
print("\nQuality Summary:")
for key, value in quality_summary.items():
    print(f"{key}: {value}")


## Observations After Quality Checks

1. **Duplicate Rows**:
   - Checked for duplicate rows and removed them from the dataset.
   - The final row count reflects the removal of duplicates.

2. **Negative Values**:
   - Verified that numeric columns (e.g., `number_tested`, `number_confirmed`, etc.) contain no negative values.

3. **Outliers**:
   - Visualized the distribution of numeric columns using a boxplot.
   - Any extreme values or potential outliers identified can be reviewed for validity.

### Next Steps:
1. Address any identified issues, such as extreme outliers or anomalies.
2. Proceed to store the cleaned and validated dataset in an SQLite database.


## Step 6: Store Data in SQLite

To make the cleaned dataset accessible for further use, we will:
1. Create (or connect to) an SQLite database file.
2. Store the dataset as a table within the database.
3. Verify that the data has been stored successfully by querying the database.

This ensures the data is saved in a structured format that can be accessed later using SQL queries or connected to other tools for analysis.


In [ ]:
# Step 1: Connect to SQLite database (or create it if it doesn't exist)
db_name = "nyc_covid_outcomes.db"  # Database file name
conn = sqlite3.connect(db_name)
print(f"Connected to SQLite database: {db_name}")

# Step 2: Store the DataFrame in the SQLite database
table_name = "covid_outcomes"
df.to_sql(table_name, conn, if_exists="replace", index=False)
print(f"Data stored in SQLite table: {table_name}")

# Step 3: Verify the stored data by querying the database
query = f"SELECT * FROM {table_name} LIMIT 5;"  # Sample query to fetch first 5 rows
sample_data = pd.read_sql_query(query, conn)

# Display the result of the query
print("\nSample Data from SQLite Database:")
print(sample_data)

# Close the connection
conn.close()
print("SQLite database connection closed.")

# Additional Explanation:
# - sqlite3.connect(): Creates or connects to the specified SQLite database file.
# - df.to_sql(): Writes the DataFrame to a table in the database. 
#   - if_exists="replace": Replaces the table if it already exists.
#   - index=False: Prevents saving the DataFrame's index as a column.
# - pd.read_sql_query(): Reads data from the database using SQL queries.


## Observations After Storing Data

1. **SQLite Database**:
   - The cleaned dataset has been successfully stored in the SQLite database file (`nyc_covid_outcomes.db`).
   - A new table (`covid_outcomes`) has been created or replaced within the database.

2. **Verification**:
   - Queried the database to retrieve a sample of rows, confirming successful storage.
  



## Step 7: Explore Data in SQLite Browser

[**Download SQLite Browser**](https://sqlitebrowser.org/) for a GUI-based exploration of the database.
   - Download and install the application.
   - Open `nyc_covid_outcomes.db` in SQLite Browser.
   - Explore the `covid_outcomes` table using the "Browse Data" tab for an easy and interactive way to inspect the data.
   - Use the "Execute SQL" tab to run SQL queries directly within the GUI.

This GUI-based tool makes it easier to interact with and analyze the database without requiring in-depth coding knowledge.



# Assignment 1 Summary: What You Just Built

## Overview:
This project demonstrated:
1. Fetching data from a public API.
2. Cleaning and transforming raw JSON data into a structured format.
3. Performing quality checks to ensure data consistency.
4. Storing the cleaned data in an SQLite database for future analysis.

## Key Steps:
### 1. Fetch Data from API:
- Used the NYC COVID-19 Outcomes API to retrieve data in JSON format.
- Implemented pagination using `$limit` and `$offset` parameters to fetch the complete dataset efficiently.

### 2. Data Cleaning:
- Converted date columns to `datetime` format.
- Converted numeric columns to integers for accurate analysis.
- Replaced missing values with `0` for consistency.

### 3. Quality Checks:
- Identified and removed duplicate rows from the dataset.
- Verified that numeric columns contained no negative values.
- Visualized numeric data distributions using boxplots to identify potential outliers.

### 4. Store Data in SQLite:
- Stored the cleaned dataset in an SQLite database (`nyc_covid_outcomes.db`) as the table `covid_outcomes`.
- Verified successful storage by querying the database.


# Questions (graded, 20 points): answer all 10 using your own full dataset

1. How many data instances (rows) are there in the data set?
2. How many features (columns) does each data instance have?
3. What are the types of each feature?
4. How was the data collected?
5. What can the data be used for?
6. What are the dates over which the data was collected?
7. What are the limitations of the data?
8. Who manages the data?
9. The **extract_date** is the date that the dataset is updated by local hospitals and clinics.The **specimen_date** is the date on which the specimen was taken from the patient at testing, not all specimens taken on a given date are uploaded to the database on the same **extract_date**. For each of these specimen dates below , What are the number tested, number of hosptilization and number of deaths.
  
          1. 22nd December 2020
          2. 30th August 2021
          3. 8th February 2021
          4. 14th May 2020
10. What are the most important concepts and skills that you have learned from this excercise? 
